# Chapter 22 — LLM Red Team 자동화 도구 — Agent 루프·안전장치

> **Chapter 22 · Part III**

자동 공격 Agent 루프에 **실행 전 예산 검사**와 실제로 작동하는 kill switch 를 걸어 통제하고, §22.3.1 Computer Use 의 관찰→판단→행동 루프에서 **화면에 주입된 악성 지시**를 대상 허용목록·kill switch 로 막습니다.

## 실습 목표
1. 목표 지향 Agent 루프를 구현합니다.
2. **비용을 쓰기 전에** 예산을 확인해 상한을 절대 넘지 않게 하고, kill switch 로 즉시 중단합니다.
3. Computer Use 스타일 관찰(`grab_screen`)→판단→행동(`apply_action`) 루프를 시뮬레이션 화면으로 체험합니다.
4. 화면에 주입된 지시(예: 관리자 페이지로 이동해 사용자 삭제)를 **대상 허용목록**과 kill switch 로 차단합니다.

> **주의:** 데모는 표준 라이브러리(+선택적으로 `anthropic`)만으로 동작하며, 위험 동작과 화면·입출력 장치는 모두 시뮬레이션입니다.

In [ ]:
# Step 2 에서 실제 Claude API 를 호출하려면 anthropic 필요 (없어도 mock 으로 동작)
# !pip install -q anthropic
# (PentestGPT 등 외부 CLI 도구는 설치하지 않음 — 이 노트북은 루프·안전장치·판단 호출만 다룸)

In [ ]:
# 공통 안전장치 — Step 1 과 Step 2 가 그대로 재사용한다
KILL = {'flag': False}

class Budget:
    """남은 예산을 들고, 쓰기 전에 감당 가능한지 확인한다."""
    def __init__(self, total):
        self.remaining = total
    def can_afford(self, cost):
        return self.remaining >= cost
    def spend(self, cost):
        self.remaining -= cost

def guards_ok(budget: 'Budget', cost: int, log: list) -> bool:
    """실행 '전에' 검사: kill switch 가 켜졌거나 남은 예산이 비용보다 적으면 실행하지 않는다."""
    if KILL['flag']:
        log.append('KILL 스위치 작동 — 즉시 중단')
        return False
    if not budget.can_afford(cost):
        log.append(f'예산 부족(남은 {budget.remaining} < 비용 {cost}) — 실행 전 중단')
        return False
    return True

---
## Step 1. 예산·kill switch 가 있는 Agent 루프

실제 red-team Agent 는 반드시 상한과 중단 장치를 둡니다(무한 공격 방지). 핵심은 **비용을
쓰기 전에** 예산을 확인하는 것입니다 — 확인을 나중에 하면 한도를 초과한 뒤에야 멈춥니다.
아래 두 실행으로 (a) 예산 상한이 초과 없이 멈추는 것과 (b) kill switch 가 실제로 즉시
중단시키는 것을 확인합니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: kill_at_step 이 지정되면 그 스텝에 도달할 때 외부 감시가 kill switch 를 켠다(시연용)

# (a) 예산 상한: 100 예산, 스텝당 30 → 3 스텝(90) 쓰고, 4번째는 남은 10<30 이라 실행 전 중단
KILL['flag'] = False
log, budget = agent_loop('recon→exploit')
for line in log:
    print(' ', line)
print(f'→ 남은 예산 {budget.remaining} (한 번도 음수가 되지 않음 — 상한을 넘지 않습니다).')
assert budget.remaining >= 0

# (b) kill switch: 2번째 스텝에서 외부 감시가 KILL 을 켜면 즉시 중단
print('\n[kill switch 시연]')
KILL['flag'] = False
log2, _ = agent_loop('recon→exploit', kill_at_step=2)
for line in log2:
    print(' ', line)
KILL['flag'] = False   # 다음 스텝을 위해 원복

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
def agent_loop(goal, max_steps=5, budget_total=100, cost=30, kill_at_step=None):
    budget = ...
    log = ...
    for step in range(max_steps):
        # kill_at_step 이 지정되면 그 스텝에 도달할 때 외부 감시가 kill switch 를 켠다(시연용)
        if kill_at_step is not None and step == kill_at_step:
            KILL['flag'] = ...
        if not guards_ok(budget, cost, log): ...
            break
        ...
    return ...
```
</details>


---
## Step 2. Computer Use 스타일 인지→판단→행동 루프 + 화면 주입 방어 (시뮬레이션)

§22.3.1 이 보여주는 Computer Use 흐름 — `screenshot = grab_screen()` → 판단 → `click`/`type`/`key` 실행 — 을 체험하고, 여기에 **화면에 주입된 악성 지시**를 방어하는 안전장치를 붙입니다.

> **주의 — 무엇이 시뮬레이션인가:** 실제 화면·마우스·키보드는 전혀 쓰지 않습니다. `grab_screen()` 은 진짜 스크린샷 대신 **가짜 UI 상태를 텍스트(JSON)로 기술**해 돌려주고, `apply_action()` 은 그 액션을 상태 전이 규칙에 반영만 합니다 — ch24 의 `nmap_scan_mock`·ch25 의 `MockCTFdServer` 와 같은 방식의 mock 입니다. 정상 목표의 "다음 단계는?" 판단은 `ANTHROPIC_API_KEY` 가 있으면 실제 Claude API 를 호출합니다(없으면 규칙 기반 mock).

> **Computer Use tool 타입에 관하여:** 본문 §22.3.1 의 `computer_20241022` 는 2024-10 첫 베타 당시의 tool 타입입니다. 이후 모델에서는 Anthropic 이 직접 호스팅하는 sandbox 기반 방식으로 정리됐습니다(본문 §22.3.1 각주 참고 — 정확한 tool 타입 이름·지원 범위는 시기에 따라 바뀌므로 원고 각주의 확인 시점 표기를 따릅니다). 그래서 이 노트북은 특정 tool 타입에 의존하지 않고, 같은 관찰→판단→행동 구조를 **커스텀 tool(`take_action`) 기반 구조화 JSON 판단**으로 흉내냅니다.

**안전장치:** Step 1 의 `Budget`·`KILL`·`guards_ok` 를 그대로 재사용하고, 추가로 **대상 허용목록**(로그인 폼 요소만 조작 허용)을 둡니다. 화면에 "관리자 페이지로 가서 사용자 삭제" 같은 지시가 주입돼 Agent 가 허용목록 밖 요소를 건드리려 하면, kill switch 가 켜지고 그 액션은 **실행되지 않습니다**.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: 대상 허용목록 — 이 요소들만 조작을 허용한다(로그인 폼). 이 밖은 차단 대상.

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
import os, json
API_KEY = ...
USE_REAL = ...
if USE_REAL:
    import anthropic
    client = ...

# 대상 허용목록 — 이 요소들만 조작을 허용한다(로그인 폼). 이 밖은 차단 대상.
ALLOWED_TARGETS = ...

# computer_20241022 를 흉내낸 커스텀 tool — 클릭/입력/키 액션을 구조화 JSON 으로 강제
TOOL_TAKE_ACTION = ...
    ...

def grab_screen(state: ...
    ...
    injected = ...
    screens = ...
        ...
    scr = ...
    if injected and state == 'login_empty':
        # 공격자가 화면에 심은 지시 배너 + 관리자 삭제 링크(허용목록 밖 요소)
        ...
    return ...

def apply_action(state: ...
    ...
    a, target, value = ...
    if state == 'login_empty' and a == 'type' and target == 'username':
        return ...
    if state == 'login_user_only' and a == 'type' and target == 'password':
        return ...
    if state == 'login_filled' and a in ('click', 'key') and (target == 'login_btn' or value.lower() == 'enter'):
        return ...
    return ...
```
</details>


In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: ANTHROPIC_API_KEY 없을 때 쓰는 규칙 기반 offline 대체 판단(정상 로그인 흐름)

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
def decide_next_action_mock(state: ...
    # ANTHROPIC_API_KEY 없을 때 쓰는 규칙 기반 offline 대체 판단(정상 로그인 흐름)
    rules = ...
        ...
    a = ...
    a['rationale'] = ...
    return ...

def decide_naive_injected(state: ...
    # 화면 주입에 넘어간 '순진한' planner 를 가정 — 배너 지시를 그대로 따른다
    return ...
            ...

def decide_next_action(goal: ...
    """관찰(screen) → 판단. 책 §22.3.1 의 'response = ...
    if not USE_REAL:
        return ...
    resp = ...
        model = ...
        thinking = ...
        system = ...
                ...
        tools = ...
        messages = ...
            f'목표: {goal}\n현재 화면(시뮬레이션):\n{json.dumps(screen, ensure_ascii = ...
    ...
    return ...

def enforce_allowlist(action: ...
    ...
    tgt = ...
    if action.get('action') != 'done' and tgt and tgt not in ALLOWED_TARGETS:
        KILL['flag'] = ...
        ...
        return ...
    return ...

def computer_use_agent_loop(goal, max_steps=6, budget_total=100, cost=20,
                            injected = ...
    ...
    state, budget, log = ...
    for step in range(max_steps):
        if not guards_ok(budget, cost, log): ...
            break
        if state == 'dashboard':
            ...
        screen = ...
        ...
        action = ...
        if not enforce_allowlist(action, log): ...
            break
        new_state = ...
        ...
        state = ...
    return ...

# 실행 1 — 정상 목표(주입 없음): 실제 Claude(키 있으면) 또는 mock 이 로그인 완료
KILL['flag'] = ...
GOAL = ...
...
for line in computer_use_agent_loop(GOAL, injected=False):
    ...

# 실행 2 — 화면에 악성 지시 주입 + 주입에 넘어간 planner: 허용목록/KILL 이 삭제 실행을 막는다
KILL['flag'] = ...
...
for line in computer_use_agent_loop(GOAL, injected=True, decider=decide_naive_injected):
    ...
KILL['flag'] = ...
```
</details>


정상 실행의 "3단계 만에 목표 도달"이 매번 보장되진 않습니다 — 판단이 실제 Claude 호출이면 클릭 위치를 한 번 더 확인하는 등 다른 순서를 고를 수도 있습니다. 어느 쪽이든 `Budget`·`max_steps`·`KILL`·허용목록이 §22.4.2 **격리 운영 4 원칙**을 그대로 구현합니다.

1. **인터넷 차단** — `grab_screen`/`apply_action` 은 네트워크·실제 화면에 전혀 접근하지 않습니다.
2. **시간·스텝 cap** — `max_steps`, 그리고 비용을 쓰기 전 검사하는 `Budget`.
3. **행동 범위 제한** — 대상 허용목록 밖 조작은 실행 전에 차단하고 kill switch 를 켭니다.
4. **사람 sample 감사** — 위 `log` 를 사람이 그대로 검토할 수 있습니다.

---
## 정리
- 미토스/페이블5 시대 핵심 위협은 특수 모델이 아니라 **일반 Agent + 자율 루프**입니다.
- 자동 공격 도구에는 반드시 스텝·비용 상한과 kill switch 를 두되, **비용을 쓰기 전에** 예산을 확인해 상한을 절대 넘지 않게 합니다.
- Computer Use 의 관찰→판단→행동 루프는 화면 자체가 공격 표면입니다 — 화면에 주입된 지시를 명령으로 따르지 않도록 하고, **대상 허용목록**과 kill switch 로 허용된 조작만 실행합니다.
- 안전장치는 판단(모델)이 속더라도 동작해야 합니다 — 허용목록은 액션이 실행되기 전 마지막 방어선입니다.

> **다음 챕터:** Frontier Safety 정책.